In [1]:
import torch
from torch import nn

class ManualLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()

        self.gamma = nn.Parameter(
            torch.ones(normalized_shape)
        )
        self.beta = nn.Parameter(
            torch.zeros(normalized_shape)
        )
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)

        var = x.var(
            dim=-1,
            keepdim=True,
            unbiased=False
        )

        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        return self.gamma * x_norm + self.beta

In [2]:
torch.manual_seed(42)

x = torch.randn(3, 4)

manual_ln = ManualLayerNorm(4)
torch_ln = nn.LayerNorm(4)

# 确保两者参数完全相同
with torch.no_grad():
    torch_ln.weight.copy_(manual_ln.gamma)
    torch_ln.bias.copy_(manual_ln.beta)

manual_output = manual_ln(x)
torch_output = torch_ln(x)

print("Manual output:\n", manual_output)
print("PyTorch output:\n", torch_output)

torch.testing.assert_close(
    manual_output,
    torch_output,
    rtol=1e-5,
    atol=1e-6
)

print("Mean:", manual_output.mean(dim=-1))
print("Variance:", manual_output.var(dim=-1, unbiased=False))

Manual output:
 tensor([[ 1.4150, -1.4102,  0.0257, -0.0305],
        [-0.9276, -0.1964,  1.6731, -0.5491],
        [-0.2914, -1.2908,  0.0853,  1.4969]], grad_fn=<AddBackward0>)
PyTorch output:
 tensor([[ 1.4150, -1.4102,  0.0257, -0.0305],
        [-0.9276, -0.1964,  1.6731, -0.5491],
        [-0.2914, -1.2908,  0.0853,  1.4969]],
       grad_fn=<NativeLayerNormBackward0>)


Mean: tensor([-9.3132e-10, -1.4901e-08,  1.7881e-07], grad_fn=<MeanBackward1>)
Variance: tensor([0.9982, 1.0000, 0.9997], grad_fn=<VarBackward0>)


In [3]:
batch_a = torch.tensor([
    [1., 2., 3., 4.],
    [2., 3., 4., 5.],
    [3., 4., 5., 6.],
    [4., 5., 6., 7.]
])

batch_b = torch.tensor([
    [1., 2., 3., 4.],
    [100., 100., 100., 100.],
    [200., 200., 200., 200.],
    [300., 300., 300., 300.]
])

ln = nn.LayerNorm(4)

bn = nn.BatchNorm1d(
    num_features=4,
    affine=False,
    track_running_stats=False
)

ln.train()
bn.train()

ln_a = ln(batch_a)
ln_b = ln(batch_b)

bn_a = bn(batch_a)
bn_b = bn(batch_b)

print("LayerNorm, batch A:", ln_a[0])
print("LayerNorm, batch B:", ln_b[0])

print("BatchNorm, batch A:", bn_a[0])
print("BatchNorm, batch B:", bn_b[0])

# LayerNorm 只看自己的特征，不受其他样本影响
torch.testing.assert_close(ln_a[0], ln_b[0])

# BatchNorm 训练时依赖整个 batch 的统计量
assert not torch.allclose(bn_a[0], bn_b[0])

LayerNorm, batch A: tensor([-1.3416, -0.4472,  0.4472,  1.3416], grad_fn=<SelectBackward0>)
LayerNorm, batch B: tensor([-1.3416, -0.4472,  0.4472,  1.3416], grad_fn=<SelectBackward0>)
BatchNorm, batch A: tensor([-1.3416, -1.3416, -1.3416, -1.3416])
BatchNorm, batch B: tensor([-1.3389, -1.3362, -1.3335, -1.3307])


## BatchNorm 小 batch 与 train/eval 行为实验

`track_running_stats=True`（默认值）时的完整行为。

In [4]:
# 实验 1: batch=1 时 BatchNorm 能否工作，取决于每个通道有多少个统计元素

# 输入 [1, C]（无空间维度）：每通道只有 1 个元素，方差无定义，训练直接报错
bn = nn.BatchNorm1d(4)
bn.train()
try:
    bn(torch.randn(1, 4))
except ValueError as e:
    print("ValueError:", e)

# 输入 [1, C, L]（有空间维度，如 Conv 特征图）：
# 统计量在 (N, L) 维联合计算，通道内仍有 L 个元素，可以训练
bn2 = nn.BatchNorm1d(4)
bn2.train()
y = bn2(torch.randn(1, 4, 5))
print("batch=1 但有空间维度，训练成功，输出 shape:", y.shape)

ValueError: Expected more than 1 value per channel when training, got input size torch.Size([1, 4])
batch=1 但有空间维度，训练成功，输出 shape: torch.Size([1, 4, 5])


In [5]:
# 实验 2: track_running_stats=True（默认）的 train/eval 行为

bn = nn.BatchNorm1d(3)
print("初始 running:", bn.running_mean, bn.running_var)

# ---- train 模式：归一化用当前 batch 统计量，同时更新 running（momentum=0.1）----
bn.train()

x1 = torch.tensor([[1., 2., 3.], [3., 4., 5.]])  # batch 均值 [2, 3, 4]
bn(x1)
print("batch1 后 running:", bn.running_mean, bn.running_var)
# running_mean = 0.9*0 + 0.1*[2, 3, 4] = [0.2, 0.3, 0.4]

x2 = torch.tensor([[5., 6., 7.], [7., 8., 9.]])  # batch 均值 [6, 7, 8]
bn(x2)
print("batch2 后 running:", bn.running_mean, bn.running_var)
# running_mean = 0.9*[0.2, 0.3, 0.4] + 0.1*[6, 7, 8] = [0.78, 0.97, 1.16]

# ---- eval 模式：不再算 batch 统计量，直接用 running 统计量 ----
bn.eval()

x_new = torch.tensor([[10., 11., 12.]])
y_eval = bn(x_new)
print("eval 输出:", y_eval.detach())

# 验证：eval 输出 == 用 running 统计量手动归一化
manual = (x_new - bn.running_mean) / torch.sqrt(bn.running_var + bn.eps)
print("running 统计量手动归一化:", manual)
print("两者一致:", torch.allclose(y_eval, manual))

# eval 下行为确定：重复调用结果一致
print("eval 重复调用一致:", torch.allclose(bn(x_new), y_eval))

初始 running: tensor([0., 0., 0.]) tensor([1., 1., 1.])
batch1 后 running: tensor([0.2000, 0.3000, 0.4000]) tensor([1.1000, 1.1000, 1.1000])
batch2 后 running: tensor([0.7800, 0.9700, 1.1600]) tensor([1.1900, 1.1900, 1.1900])
eval 输出: tensor([[8.4519, 9.1944, 9.9370]])
running 统计量手动归一化: tensor([[8.4519, 9.1944, 9.9370]])
两者一致: True
eval 重复调用一致: True
